## 1. Setup

In [0]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import mlflow
from mlflow.models import infer_signature
from sklearn import metrics
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

GOLD = "capstone_team01.gold_dq"
EXPERIMENT = "/Users/datauser43@financecloudbrick.onmicrosoft.com/card_fraud_detection_experiment"
SEED = 42
THRESHOLD = 0.5   # probability above which we call a transaction suspicious

mlflow.set_experiment(EXPERIMENT)

## 2. Load data

In [0]:
# One row per transaction: the label + features known at the moment of the transaction
pdf = spark.sql(f"""
SELECT
  f.transaction_id,
  f.transaction_ts,
  CASE WHEN f.current_status = 'REVERSED' OR f.failed_auth_count > 0 THEN 1 ELSE 0 END AS label,

  -- numeric features
  log1p(f.amount)                                          AS log_amount,
  f.transaction_hour,
  CAST(dayofweek(f.transaction_ts) IN (1, 7) AS INT)       AS is_weekend,
  CAST(f.is_cross_border AS INT)                           AS is_cross_border,
  c.daily_limit,
  datediff(f.transaction_ts, c.issue_date)                 AS card_age_days,
  datediff(f.transaction_ts, cu.onboarded_date)            AS customer_tenure_days,
  f.card_txn_count_1h,
  f.card_txn_count_24h,
  f.card_distinct_merchants_24h,
  f.customer_txn_count_24h,
  f.minutes_since_prev_card_txn,

  -- categorical features
  f.currency, f.channel, f.country, m.merchant_category, f.merchant_risk_band,
  cu.customer_segment, f.customer_risk_band, c.card_type

FROM {GOLD}.fact_card_transaction f
JOIN {GOLD}.dim_customer cu ON f.customer_id = cu.customer_id
JOIN {GOLD}.dim_merchant m  ON f.merchant_id = m.merchant_id
JOIN {GOLD}.dim_card     c  ON f.card_id     = c.card_id
""").toPandas()

NUM = ["log_amount", "transaction_hour", "is_weekend", "is_cross_border", "daily_limit", "card_age_days",
       "customer_tenure_days", "card_txn_count_1h", "card_txn_count_24h", "card_distinct_merchants_24h",
       "customer_txn_count_24h", "minutes_since_prev_card_txn"]
CAT = ["currency", "channel", "country", "merchant_category", "merchant_risk_band",
       "customer_segment", "customer_risk_band", "card_type"]
FEATURES = NUM + CAT
pdf[NUM] = pdf[NUM].astype(float)

print("Rows:", len(pdf), "| features:", len(FEATURES))
display(pdf.head())

## 3. Explore the data

In [0]:
# Feature types and missing values
display(pd.DataFrame({
    "feature": FEATURES,
    "type": ["numeric" if c in NUM else "categorical" for c in FEATURES],
    "missing_%": (pdf[FEATURES].isna().mean() * 100).round(2).values}))
display(pdf[NUM + ["label"]].describe().T)

# Three quick charts
fig, axes = plt.subplots(1, 3, figsize=(18, 4))
pdf.label.value_counts().sort_index().plot.bar(ax=axes[0], title="Class balance (0 = normal, 1 = suspicious)")
pdf.groupby("channel").label.mean().plot.bar(ax=axes[1], title="Suspicious rate by channel")
pdf[NUM].corrwith(pdf.label).sort_values().plot.barh(ax=axes[2], title="Correlation of numeric features with label")
plt.tight_layout()
plt.show()

print("Share of suspicious transactions:", round(pdf.label.mean(), 4))

## 4. Split by time

In [0]:
# Oldest 70% = train, next 15% = validation, newest 15% = test (we predict future transactions)
pdf = pdf.sort_values("transaction_ts").reset_index(drop=True)
n = len(pdf)
train = pdf.iloc[: int(n * 0.70)]
valid = pdf.iloc[int(n * 0.70): int(n * 0.85)]
test  = pdf.iloc[int(n * 0.85):]

for name, part in [("train", train), ("validation", valid), ("test", test)]:
    print(f"{name:10s} rows={len(part):5d}  {part.transaction_ts.min()} -> {part.transaction_ts.max()}  "
          f"suspicious={part.label.mean():.3f}")

## 5. Train two models and track them in MLflow

In [0]:
def make_pipeline(model):
    """Fill missing values, scale numbers, one-hot encode categories, then the model."""
    prep = ColumnTransformer([
        ("num", Pipeline([("impute", SimpleImputer()), ("scale", StandardScaler())]), NUM),
        ("cat", Pipeline([("impute", SimpleImputer(strategy="most_frequent")),
                          ("onehot", OneHotEncoder(handle_unknown="ignore"))]), CAT)])
    return Pipeline([("prep", prep), ("model", model)])

# class_weight="balanced" lets a 0.5 threshold work even though suspicious rows are the minority
models = {
    "logistic_regression": LogisticRegression(max_iter=1000, class_weight="balanced"),
    "random_forest": RandomForestClassifier(n_estimators=300, min_samples_leaf=20, class_weight="balanced",
                                            random_state=SEED, n_jobs=-1),
}

results, pipes = [], {}
for name, model in models.items():
    with mlflow.start_run(run_name=name) as run:
        pipe = make_pipeline(model).fit(train[FEATURES], train.label)
        p = pipe.predict_proba(valid[FEATURES])[:, 1]
        scores = {"valid_roc_auc": metrics.roc_auc_score(valid.label, p),
                  "valid_pr_auc": metrics.average_precision_score(valid.label, p),
                  "valid_f1": metrics.f1_score(valid.label, p >= THRESHOLD)}
        mlflow.log_params({"model": name, "seed": SEED, "threshold": THRESHOLD, "n_train": len(train)})
        mlflow.log_metrics(scores)
    pipes[name] = pipe
    results.append({"model": name, "run_id": run.info.run_id, **scores})

results = pd.DataFrame(results).sort_values("valid_pr_auc", ascending=False)
display(results.round(4))
best_name, best_run_id = results.model.iloc[0], results.run_id.iloc[0]
print("Best model on validation:", best_name)

## 6. Test the best model on the newest data

In [0]:
best = pipes[best_name]
p_test = best.predict_proba(test[FEATURES])[:, 1]
y_pred = (p_test >= THRESHOLD).astype(int)

test_scores = {"test_roc_auc": metrics.roc_auc_score(test.label, p_test),
               "test_pr_auc": metrics.average_precision_score(test.label, p_test),
               "test_accuracy": metrics.accuracy_score(test.label, y_pred),
               "test_precision": metrics.precision_score(test.label, y_pred),
               "test_recall": metrics.recall_score(test.label, y_pred),
               "test_f1": metrics.f1_score(test.label, y_pred)}

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
metrics.ConfusionMatrixDisplay.from_predictions(test.label, y_pred, display_labels=["normal", "suspicious"],
                                                ax=axes[0], colorbar=False)
metrics.RocCurveDisplay.from_predictions(test.label, p_test, ax=axes[1])
axes[1].plot([0, 1], [0, 1], "k--")   # random-guess line
plt.tight_layout()

with mlflow.start_run(run_id=best_run_id):
    mlflow.log_metrics(test_scores)
    mlflow.log_figure(fig, "test_evaluation.png")
plt.show()

display(pd.DataFrame([test_scores]).round(4))
print("ROC-AUC 0.5 = random guessing. Baseline PR-AUC (share of suspicious):", round(test.label.mean(), 4))

## 7. Retrain on all data and save the final model

In [0]:
final_model = make_pipeline(models[best_name]).fit(pdf[FEATURES], pdf.label)
signature = infer_signature(pdf[FEATURES].head(100), final_model.predict_proba(pdf[FEATURES].head(100)))

with mlflow.start_run(run_name=f"final_{best_name}") as final_run:
    mlflow.log_params({"model": best_name, "threshold": THRESHOLD, "n_final_fit": len(pdf),
                       "n_train": len(train), "n_test": len(test)})
    mlflow.log_metrics(test_scores)   # holdout scores from step 6
    model_info = mlflow.sklearn.log_model(final_model, name="model", signature=signature,
                                          input_example=pdf[FEATURES].head(3),
                                          pyfunc_predict_fn="predict_proba")   # loaded model returns probabilities

print("Model saved:", model_info.model_uri)

## 8. Score transactions and compare with actual labels

In [0]:
# Load the saved model back from MLflow (like a batch job would) and score every transaction
loaded = mlflow.pyfunc.load_model(model_info.model_uri)
pdf["fraud_probability"] = np.asarray(loaded.predict(pdf[FEATURES]))[:, 1]
pdf["predicted"] = (pdf.fraud_probability >= THRESHOLD).astype(int)

# Actual vs predicted counts (note: includes rows the final model was trained on; step 6 is the honest test)
display(pd.crosstab(pdf.label, pdf.predicted, rownames=["actual"], colnames=["predicted"], margins=True)
        .reset_index())
display(pdf.sort_values("fraud_probability", ascending=False)
        [["transaction_id", "transaction_ts", "fraud_probability", "predicted", "label"]].head(10))

# Optional: save scores to a table
SAVE = False
if SAVE:
    spark.createDataFrame(pdf[["transaction_id", "fraud_probability", "predicted"]]) \
         .write.mode("overwrite").saveAsTable(f"{GOLD}.fraud_model_scores")